## Import Libraries

In [52]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

## data generator 

### 1a. Univariate  gaussian:
#### Using Box–Muller transform  
ref: https://en.wikipedia.org/wiki/Box%E2%80%93Muller_transform

In [53]:
class BoxMullerTransform():
    def __init__(self, mean, variance) :
        self.mean = mean
        self.variance = variance
        self.std_dev = np.sqrt(variance)
    def sample_standard_normal(self):
        u1 = np.random.uniform(0, 1)
        u2 = np.random.uniform(0, 1)
        self.x1 = np.sqrt(-2 * np.log(u1)) * np.cos(2 *  np.pi * u2)
        self.x2 = np.sqrt(-2 * np.log(u1)) * np.sin(2 * np.pi * u2)
    def sample_normal(self):
        self.sample_standard_normal()
        p = np.random.random()
        if p < 0.5:
            x = self.x1
        else:
            x = self.x2
        y = self.mean + self.std_dev * x
        
        return y


### 1b. Polynomial  basis  linear  model  data  generator


In [54]:
class PolynomialBasisLinearModel():
    def __init__(self, n, a, w):
        self.n = n
        self.variance = a
        self.weight = np.array(w)
        self.box_muller = BoxMullerTransform(0, a)
    def get_X(self, x):
        X = []
        for i in range(self.n):
            X.append(x ** i)
        return X
    def get_sample(self):
        x = np.random.uniform(-1, 1)
        X = np.array(self.get_X(x))
        e = self.box_muller.sample_normal()
        y = np.transpose(self.weight) @ X + e
        return x, y

## 2. Sequential Estimator
#### Using Welford's online algorithm  
ref: https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Online_algorithm

In [55]:
class SequentialEstimator():
    def __init__(self, m, s):
        self.mean = float(m)
        self.variance = float(s)
        self.estimate_mean = 0.0
        self.estimate_variance = 0.0
        self.M2 = 0.0
        self.n = 0.0
        self.gassian_data_generator = BoxMullerTransform(self.mean, self.variance)
    def get_new_data(self):
        self.n = self.n + 1
        return self.gassian_data_generator.sample_normal()
    def update_mean_and_M2(self, new_data):
        delta_1 = new_data - self.estimate_mean
        self.estimate_mean += delta_1 / self.n
        delta_2 = new_data - self.estimate_mean
        self.M2 += delta_1 * delta_2
    def estimate(self):
        print(f"Data point source function: N({self.mean}, {self.variance})\n")
        while abs(self.mean - self.estimate_mean) >= 0.01 or abs(self.variance - self.estimate_variance) >= 0.01:
            new_x = self.get_new_data()
            self.update_mean_and_M2(new_x)
            if self.n == 1:
                self.estimate_variance = 0
            else :
                self.estimate_variance = self.M2 / (self.n - 1)
            print(f"Add data point: {new_x}")
            print(f"Mean = {self.estimate_mean} Variance = {self.estimate_variance}")
        

#### Experiment with mean = 3 and variance = 5

In [ ]:
sequential_estimator = SequentialEstimator(3., 5.)
sequential_estimator.estimate()

## Baysian Linear regression

#### Plotter is for plotting the output of the regression

In [57]:
class Plotter:
    def __init__(self, w, a):
        self.w = w
        self.a = a
        self.fig, self.axs = plt.subplots(2, 2, figsize=(10, 8))
        self.plot_count = 0
        self.x_line = np.linspace(-2, 2, 1000)
        self.plot_ground_truth(1 / a)
        self.count = 0
    
    def plot_ground_truth(self, a):
        coeffs = self.w[::-1]
        polynomial = np.poly1d(coeffs)
        y = polynomial(self.x_line)
        y1 = polynomial(self.x_line) + a
        y2 = polynomial(self.x_line) - a
        self.axs[0, 0].scatter(self.x_line, y, color = 'black', s=1)
        self.axs[0, 0].scatter(self.x_line, y1, color = 'red', s=1)
        self.axs[0, 0].scatter(self.x_line, y2, color = 'red', s=1)
        self.axs[0, 0].set_title('Ground truth')
        self.axs[0, 0].set_xlim(-2, 2)
        self.axs[0, 0].set_ylim(-20, 20)
        self.axs[0, 0].xaxis.set_major_locator(MultipleLocator(1))
        self.axs[0, 0].yaxis.set_major_locator(MultipleLocator(10))
    def get_variance(self, X, Lambda):
        return (1 / self.a + X @ np.linalg.inv(Lambda) @ np.transpose(X))
    def plot_sub(self, data_x, data_y, w, Lambda, i, n):
        self.count += 1
        coeffs = w[::-1]
        polynomial = np.poly1d(coeffs)
        x_variance = []
        y = polynomial(self.x_line)
        x_transformed = np.column_stack([self.x_line ** k for k in range(n)])
        for row in x_transformed:
            x_variance.append(self.get_variance(row, Lambda))
        y1 = polynomial(self.x_line) + x_variance
        y2 = polynomial(self.x_line) - x_variance
        print(y1.shape, y2.shape)
        self.axs[1, self.count - 1].scatter(self.x_line, y, color = 'black', s = 1)
        self.axs[1, self.count - 1].scatter(self.x_line, y1, color = 'red', s = 1)
        self.axs[1, self.count - 1].scatter(self.x_line, y2, color = 'red', s = 1)
        self.axs[1, self.count - 1].scatter(data_x, data_y, color = 'blue', s = 15, alpha = 0.3)
        self.axs[1, self.count - 1].set_title(f'After {i} incomes')
        self.axs[1, self.count - 1].set_xlim(-2, 2)
        self.axs[1, self.count - 1].set_ylim(-20, 20)
        self.axs[1, self.count - 1].xaxis.set_major_locator(MultipleLocator(1))
        self.axs[1, self.count - 1].yaxis.set_major_locator(MultipleLocator(10))
        
    def plot_all(self, data_x, data_y, w, Lambda, n):
        self.count += 1
        coeffs = w[::-1]
        polynomial = np.poly1d(coeffs)
        x_variance = []
        y = polynomial(self.x_line)
        x_transformed = np.column_stack([self.x_line ** k for k in range(n)])
        for row in x_transformed:
            x_variance.append(self.get_variance(row, Lambda))
        y1 = polynomial(self.x_line) + x_variance
        y2 = polynomial(self.x_line) - x_variance
        self.axs[0, 1].scatter(self.x_line, y, color = 'black', s = 1)
        self.axs[0, 1].scatter(self.x_line, y1, color = 'red', s = 1)
        self.axs[0, 1].scatter(self.x_line, y2, color = 'red', s = 1)
        self.axs[0, 1].scatter(data_x, data_y, color = 'blue', s = 15, alpha=0.3)
        self.axs[0, 1].set_title('Predict result')
        self.axs[0, 1].set_xlim(-2, 2)
        self.axs[0, 1].set_ylim(-20, 20)
        self.axs[0, 1].xaxis.set_major_locator(MultipleLocator(1))
        self.axs[0, 1].yaxis.set_major_locator(MultipleLocator(10))
        plt.tight_layout()
        plt.show()

#### Basian Linear regression model

In [58]:
class BaysianLinearRegression():
    def __init__(self, b, n, a, w):
        self.n = n
        self.polynomial_data_generator = PolynomialBasisLinearModel(self.n, a, w)
        self.prior_mean_matrix = np.zeros((self.n, 1))
        self.prior_covariance_matrix = np.eye(self.n) * (1. / b)
        self.y_mean = 0
        self.y_variance = 1
        self.a = 1 / a
        self.iter = 0
        self.plot_x = []
        self.plot_y = []
        self.plotter = Plotter(w, self.a)
    def get_sample_point(self):
        x, y = self.polynomial_data_generator.get_sample()
        return x, y
    def get_X(self, x):
        X = []
        for i in range(self.n): 
            X.append(x ** i)
        return X
    def update_parameters(self, X, y):
        self.iter += 1
        self.Lambda = (self.a * np.transpose(X) @ X ) + np.linalg.inv(self.prior_covariance_matrix)
        self.prior_mean_matrix = np.linalg.inv(self.Lambda) @ (self.a * np.transpose(X) * y + np.linalg.inv(self.prior_covariance_matrix) @ self.prior_mean_matrix)
        self.prior_covariance_matrix = np.linalg.inv(self.Lambda)
        variance = self.y_variance
        self.y_mean = X @ self.prior_mean_matrix 
        self.y_variance = 1 / self.a + X @ np.linalg.inv(self.Lambda) @ np.transpose(X)
        return abs(variance - self.y_variance)
    def get_prediction(self):
        x, y = self.get_sample_point()
        self.plot_x.append(x)
        self.plot_y.append(y)
        print(f"Add data point ({x: .5f}, {y: .5f}):\n")
        print()
        X = np.array(self.get_X(x)).reshape(1, -1)
        variance_diff = self.update_parameters(X, y)
        return variance_diff
    def get_output(self):
        difference = 0.01
        times = 0
        tolerance = 50
        variance_diff = float('inf')
        plot_iter = [10, 50]
        # if both the difference of variance not larger than 0.001, then model converge.
        while times <= tolerance:
            if self.iter in plot_iter:
                self.plotter.plot_sub(self.plot_x, self.plot_y, self.prior_mean_matrix.reshape(-1), self.Lambda, self.iter, self.n)
            variance_diff = self.get_prediction()
            print(f"Posterior mean:\n")
            for i in range(self.n):
                print(f'    {self.prior_mean_matrix[i].item()}')
            print(f'\nPosterior variance:\n')
            for i in range(self.prior_covariance_matrix.shape[0]):
                row_str = ', '.join(f"{self.prior_covariance_matrix[i, j]:.10f}" for j in range(self.prior_covariance_matrix.shape[1]))
                print(f"{row_str}")
            print(f'\nPredictive distribution ~ N({self.y_mean.item(): .5f}, {self.y_variance.item(): .5f})\n')
            if abs(variance_diff < difference):
                times += 1
            else:
                times = 0
        self.plotter.plot_all(self.plot_x, self.plot_y, self.prior_mean_matrix.reshape(-1), self.Lambda, self.n)
        

#### Experimeant with different cases

case1 : b = 1, n = 4, a = 1, w = [1, 2, 3, 4] 

In [ ]:
baysain_linear_regression_case1 = BaysianLinearRegression(1, 4, 1, [1, 2, 3, 4])
baysain_linear_regression_case1.get_output()

case2 : b = 100, n = 4, a = 1, w = [1, 2, 3, 4]

In [ ]:
baysain_linear_regression_case2= BaysianLinearRegression(100, 4, 1, [1, 2, 3, 4])
baysain_linear_regression_case2.get_output()

case3 : b = 1, n = 3, a = 3, w = [1, 2, 3] 

In [ ]:
baysain_linear_regression_case3= BaysianLinearRegression(1, 3, 3, [1, 2, 3])
baysain_linear_regression_case3.get_output()